# ASX 200 12-1 Momentum Backtest (Equity Signal Lab)

A test backtest of classic 12-1 price momentum on the S&P/ASX 200, built with Equity Signal Lab (BQESL)
following the workflow in the *Creating a Supply Chain Stock Selection Strategy* and
*Backtesting a Stock Selection Strategy* guides (`docs/BQuant_Help_md`).

| Setting | Value |
| --- | --- |
| Universe | S&P/ASX 200 (`AS51 Index`) members via `BQLIndexUniverse` |
| Benchmark | ASX 200 index weights (`BQLIndexWeight`) |
| Signal | Compounded total return from t−12 months to t−1 month (custom, `SignalFactory.from_user`) |
| Transformations | `winsorize(limits=(-1.5, 1.5), method='iqr')` → `zscore()` → `beta_industry_classification_neutralize` (BICS Level 1, BQL default beta) |
| Portfolios | (1) `quantile_long_only` and (2) `quantile_long_short`, quintiles, equal weight |
| Rebalance | Month end (`rebalance_freq="M"`), `implementation_lag=1` |
| Period | 2006-08-31 to 2026-08-31 (data from 2005-07-01 for the 12-month lookback) |
| Currency | AUD total returns (gross dividends) |
| Costs | `LinearTransactionCostModel(cost_in_bps=5)` |
| Reports | Performance, QuantileAnalytics, DescriptiveStatistics, Correlation, StandalonePerformance, RiskExposure, Distribution, Universe |

**Prerequisites**
- Equity Signal Lab add-on entitlement.
- Historical BICS sector names enabled for your account (request via HELP HELP, mentioning BQuant); the docs note these are not available by default.
- Point-in-time index membership for `AS51 Index` depends on your data entitlements.

In [ ]:
import numpy as np
import pandas as pd
import bql

from bloomberg.bquant.signal_lab.data_workbench import (
    create_data_pack,
    load_data_pack,
    BQLData,
    BQLIndexUniverse,
    BQLIndexWeight,
    CustomData,
    DataPackConfig,
    StorageType,
    TradingCalendarProxy,
)
from bloomberg.bquant.signal_lab.workflow import (
    AnalyticsConfig,
    AnalyticsDataConfig,
    build_backtest,
)
from bloomberg.bquant.signal_lab.workflow.node import (
    industry_grouping,
    transformers,
    combinators,
    portfolio_construction as pf_constructors,
)
from bloomberg.bquant.signal_lab.workflow.factory import (
    UniverseFactory,
    DataItemFactory,
    SignalFactory,
)
from bloomberg.bquant.signal_lab.signal.transformers import WeightingScheme
from bloomberg.bquant.signal_lab.workflow.utils import get_sandbox_path
from bloomberg.bquant.signal_lab.analytics.core.utils import LinearTransactionCostModel

bq = bql.Service()

In [ ]:
# Backtest parameters
INDEX = "AS51 Index"                           # S&P/ASX 200
CURRENCY = "AUD"

BACKTEST_START = pd.Timestamp("2006-08-31")
BACKTEST_END = pd.Timestamp("2026-08-31")
DATA_START = pd.Timestamp("2005-07-01")        # >= 12 months before BACKTEST_START for the lookback

NUM_QUANTILES = 5
WEIGHTING = WeightingScheme.EQUAL
REBALANCE_FREQ = "M"                           # calendar month end
IMPLEMENTATION_LAG = 1                         # business days
TCOST_BPS = 5

# Units of day_to_day_tot_return_gross_dvds. The docs do not state them; the Bloomberg
# field is normally in percent (1.5 = 1.5%). Verified against the data in step 1.
RETURNS_IN_PERCENT = True

DATA_PACK_PATH = f"{get_sandbox_path()}/asx200_momentum_12_1/data_pack"
CREATE_DATA_PACK = True                        # set False to reuse an existing DataPack

REPORTS = [
    "PerformanceReport",
    "QuantileAnalyticsReport",
    "DescriptiveStatisticsReport",
    "CorrelationReport",
    "StandalonePerformanceReport",
    "RiskExposureReport",
    "DistributionReport",
    "UniverseReport",
]

## 1. Create and fetch the DataPack

The DataPack snapshots the universe and data items into your User Sandbox Storage so that backtests
run on consistent data. Definitions follow the Supply Chain guide (`beta` weekly with rolling dates,
BICS names with rolling dates). The fetch covers 20+ years of ASX 200 members, so it takes a while;
if it is interrupted, run `run_fetch()` again to fetch only the missing data.

In [ ]:
data_pack_config = DataPackConfig(DATA_START, BACKTEST_END)

universe_definitions = {
    INDEX: BQLIndexUniverse(INDEX),
}

data_item_definitions = {
    "trading_calendar": TradingCalendarProxy(),
    "index_weight": BQLIndexWeight(),
    "total_return": BQLData(bq.data.day_to_day_tot_return_gross_dvds(currency=CURRENCY)),
    "beta": BQLData(bq.data.beta(), rolling_date=True, freq="w"),
    "bics_level_1": BQLData(bq.data.classification_name("bics", "1"), rolling_date=True),
    "bics_level_2": BQLData(bq.data.classification_name("bics", "2"), rolling_date=True),
    "bics_level_3": BQLData(bq.data.classification_name("bics", "3"), rolling_date=True),
    "bics_level_4": BQLData(bq.data.classification_name("bics", "4"), rolling_date=True),
}

if CREATE_DATA_PACK:
    data_pack = create_data_pack(
        data_pack_config,
        universe_definitions,
        data_item_definitions,
        StorageType.PARQUET,
        DATA_PACK_PATH,
        overwrite_existing_data_pack=True,
    )
else:
    data_pack = load_data_pack(StorageType.PARQUET, DATA_PACK_PATH)

In [ ]:
data_pack.run_fetch()
data_pack.get_fetch_status()

### Check return units

The signal compounds daily returns, so the units must be right. Typical daily equity returns are
around 1% in absolute size: roughly `1.0` if the field is in percent, `0.01` if it is a decimal.

In [ ]:
raw_returns = data_pack.get_data_item("total_return").df()
values = raw_returns.select_dtypes("number").abs().stack()
median_abs = values[values > 0].median()
print(f"median |daily total return| = {median_abs:.4f}")

looks_like_percent = median_abs > 0.1
assert looks_like_percent == RETURNS_IN_PERCENT, (
    f"RETURNS_IN_PERCENT={RETURNS_IN_PERCENT} but the data looks like "
    f"{'percent' if looks_like_percent else 'decimals'}; fix the parameter."
)

### Fill missing BICS sector names

`beta_industry_classification_neutralize` requires string sector names and says to replace null or
missing values with placeholder strings. This stores a filled copy of BICS Level 1 in the DataPack as
`CustomData` (the same update pattern as the Supply Chain guide).

In [ ]:
bics_1_raw = data_pack.get_data_item("bics_level_1").df()
text_columns = bics_1_raw.select_dtypes("object").columns
bics_1_filled = bics_1_raw.fillna({col: "Unclassified" for col in text_columns})
print(f"filled {int(bics_1_raw[text_columns].isna().sum().sum())} missing sector names")

data_pack.update_data_item_definitions(
    {"bics_level_1_filled": CustomData(dataframe=bics_1_filled)},
    overwrite=True,
)
data_pack.run_fetch()

## 2. Load the universe and data items

In [ ]:
universe = UniverseFactory.from_data_pack(
    universe_name=INDEX,
    data_pack_path=DATA_PACK_PATH,
    trading_calendar_data_item_id="trading_calendar",
    start=BACKTEST_START,
    end=BACKTEST_END,
    label="ASX 200",
)

benchmark = DataItemFactory.from_data_pack(
    universe_name=INDEX,
    data_pack=DATA_PACK_PATH,
    data_item_id="index_weight",
    start=BACKTEST_START,
    end=BACKTEST_END,
    label="Benchmark",
)

trading_calendar = DataItemFactory.from_data_pack(
    data_pack=DATA_PACK_PATH,
    data_item_id="trading_calendar",
    universe_name=INDEX,
    start=BACKTEST_START,
    end=BACKTEST_END,
)

# Returns over the backtest window, for portfolio construction and analytics
total_return = DataItemFactory.from_data_pack(
    data_pack=DATA_PACK_PATH,
    data_item_id="total_return",
    date_column="DATE",
    start=BACKTEST_START,
    end=BACKTEST_END,
)

# Returns including the 12-month lookback, for the signal
total_return_history = DataItemFactory.from_data_pack(
    data_pack=DATA_PACK_PATH,
    data_item_id="total_return",
    date_column="DATE",
    start=DATA_START,
    end=BACKTEST_END,
)

beta = DataItemFactory.from_data_pack(
    data_item_id="beta",
    data_pack=DATA_PACK_PATH,
    date_column="ITERATION_DATE",
    start=BACKTEST_START,
    end=BACKTEST_END,
    sampling_frq="w",
    fillna_method="ffill",
)

bics = {
    level: DataItemFactory.from_data_pack(
        data_item_id=f"bics_level_{level}",
        data_pack=DATA_PACK_PATH,
        date_column="ITERATION_DATE",
        start=BACKTEST_START,
        end=BACKTEST_END,
    )
    for level in (1, 2, 3, 4)
}

bics_level_1_filled = DataItemFactory.from_data_pack(
    data_item_id="bics_level_1_filled",
    data_pack=DATA_PACK_PATH,
    date_column="ITERATION_DATE",
    start=BACKTEST_START,
    end=BACKTEST_END,
)

bics_mapping = industry_grouping.build_industry_classification_mapping(
    **{
        "BICS Sector Name Level 1": bics[1],
        "BICS Industry Group Level 2": bics[2],
        "BICS Industry Level 3": bics[3],
        "BICS Sub-Industry Level 4": bics[4],
    }
)

## 3. Define the 12-1 momentum signal

The built-in `momentum` signal sums returns; this custom signal compounds them. For each date *t*
it is the total return from *t* − 12 months to *t* − 1 month, so the most recent month is excluded.
A security is scored only once it has returns covering the whole 12-month lookback.

In [ ]:
def momentum_12_1(total_returns, returns_in_percent):
    """Compounded 12-1 momentum.

    For each date t: the total return compounded from t - 12 months to t - 1 month,
    i.e. the last 12 months excluding the most recent month.

    total_returns: daily total returns, DATE index x security ID columns.
    returns_in_percent: True if 1.5 means 1.5% (divide by 100), False if 0.015 means 1.5%.
    """
    daily = total_returns / 100.0 if returns_in_percent else total_returns
    dates = daily.index

    # Cumulative growth of 1 unit. A missing day inside a security's history
    # (e.g. a trading halt) is treated as a zero return.
    growth = (1.0 + daily.fillna(0.0)).cumprod()

    def growth_as_of(offset):
        # Growth on the last available date on or before (t - offset).
        lagged = growth.reindex(dates - offset, method="ffill")
        lagged.index = dates
        return lagged

    signal = growth_as_of(pd.DateOffset(months=1)) / growth_as_of(pd.DateOffset(months=12)) - 1.0

    # Only score a security once it has returns covering the full 12-month lookback.
    first_return = daily.apply(pd.Series.first_valid_index)
    lookback_start = pd.Series(dates - pd.DateOffset(months=12), index=dates)
    has_history = pd.DataFrame(
        {sec: lookback_start >= first for sec, first in first_return.items()}, index=dates
    )
    return signal.where(has_history)


momentum_signal = SignalFactory.from_user(
    user_func=momentum_12_1,
    start=BACKTEST_START,
    end=BACKTEST_END,
    label="momentum_12_1",
    total_returns=total_return_history,
    returns_in_percent=RETURNS_IN_PERCENT,
)
my_signals = [momentum_signal]

## 4. Transformations and combination

Winsorize and z-score as in the guides, then neutralize to beta within BICS Level 1 sectors.
With a single signal the equal-weight combinator passes it through unchanged; it is included
because every documented backtest defines a combination step.

In [ ]:
my_transformers = [
    transformers.winsorize(limits=(-1.5, 1.5), method="iqr"),
    transformers.zscore(),
    transformers.beta_industry_classification_neutralize(
        beta=beta,
        industry_classification=bics_level_1_filled,
        normalize_signal=True,
    ),
]

my_signal_combination = combinators.equal_weight_combinator()

## 5. Portfolio construction, analytics and backtests

A backtest takes one portfolio constructor, so the long-only and long-short portfolios are two
backtests over the same signal, each with its own sandbox namespace.

Month-end rebalance dates that fall on a non-business day trade on the next business day
(per the `rebalance_freq` docs), and `implementation_lag=1` trades one business day after the signal date.

In [ ]:
portfolio_kwargs = dict(
    total_returns=total_return,
    trading_calendar=trading_calendar,
    implementation_lag=IMPLEMENTATION_LAG,
    num_quantiles=NUM_QUANTILES,
    weighting_scheme=WEIGHTING,
    rebalance_freq=REBALANCE_FREQ,
)
portfolios = {
    "long_only": pf_constructors.quantile_long_only(**portfolio_kwargs),
    "long_short": pf_constructors.quantile_long_short(**portfolio_kwargs),
}

analytics_config = AnalyticsConfig(
    tcost_model=LinearTransactionCostModel(cost_in_bps=TCOST_BPS)
)
analytics_data_config = AnalyticsDataConfig(
    beta=beta,
    industry_classification_mapping=bics_mapping,
    returns=total_return,
    trading_calendar=trading_calendar,
)

backtests = {
    name: build_backtest(
        universe=universe,
        benchmark_universe=benchmark,
        start=BACKTEST_START,
        end=BACKTEST_END,
        namespace=f"asx200_momentum_12_1_{name}",
        signals=my_signals,
        transformers=my_transformers,
        signal_combination=my_signal_combination,
        portfolio_construction=construction,
        reports=REPORTS,
        analytics_data_config=analytics_data_config,
        analytics_config=analytics_config,
    )
    for name, construction in portfolios.items()
}

In [ ]:
# Validate the configuration before running: signals and computation graph
backtests["long_only"].signals

In [ ]:
backtests["long_only"].visualize_graph()

## 6. Run the backtests

In [ ]:
results = {name: backtest.evaluate_graph() for name, backtest in backtests.items()}

In [ ]:
# Sanity check: 12-1 returns should mostly lie between -100% and a few hundred percent
raw_signal = results["long_only"].signals["momentum_12_1"].read()
raw_signal.stack().describe()

## 7. Analyze the results

In [ ]:
results["long_only"].plot_analytics()

In [ ]:
results["long_short"].plot_analytics()